In [1]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks
from imblearn.under_sampling import NearMiss
from imblearn.over_sampling import ADASYN

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

kfold_num =6 #7
seed_num=18
early_stop=5 #1
import random
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(seed_num)

### 데이터 읽어오기

In [ ]:
train_data = pd.read_csv("./train.csv")
test_data = pd.read_csv("./test.csv")
submission = pd.read_csv("./submission.csv")

train_data.columns = train_data.columns.str.replace(' ', '_')
train_data.columns = train_data.columns.str.replace('(', '_')
train_data.columns = train_data.columns.str.replace(')', '_')
train_data.columns = train_data.columns.str.lower()
test_data.columns = test_data.columns.str.replace(' ', '_')
test_data.columns = test_data.columns.str.replace('(', '_')
test_data.columns = test_data.columns.str.replace(')', '_')
test_data.columns = test_data.columns.str.lower()

# 결측치를 가진 컬럼 항목
na_columns = [col for col in train_data.columns if col != 'target' and train_data[col].isnull().any()]

# train_data에서의 정보를 이용하여 두 데이터에서 결측 컬럼들 삭제하기
for df in (train_data, test_data):
    df.drop(na_columns, axis=1, inplace=True)

# train_data에 'Set ID'가 없어서 test_data에서도 지우기(?) (문제가 발생한 원인일듯?)
test_data.drop(['set_id'], axis=1, inplace=True)

# 컬럼별 유니크 값의 개수가 1인 컬럼 찾기
unique_counts = train_data.nunique()
columns_with_one_unique_value = unique_counts[unique_counts == 1].index.tolist()

# train_data에서의 유니크 값의 개수가 1인 컬럼 항목들을 test_data에도 적용 (dataleakage아님)
train_data.drop(columns_with_one_unique_value, axis=1, inplace=True)
test_data.drop(columns_with_one_unique_value, axis=1, inplace=True)

# 'stage1_circle1_distance_speed_collect_result_dam', 'thickness_1_collect_result_dam', 'workmode_collect_result_dam'
# 섞인 데이터 복구
for df in (train_data, test_data):

    stage1_circle_1_1 = df[['stage1_circle1_distance_speed_collect_result_dam']].copy()
    workmode_collect_1 = df[['workmode_collect_result_dam']].copy()
    stage1_circle_1_2 = df[['stage1_circle1_distance_speed_collect_result_dam']].copy()
    workmode_collect_2 = df[['workmode_collect_result_dam']].copy()

    for i in range(len(df)):
        if stage1_circle_1_1['stage1_circle1_distance_speed_collect_result_dam'].iloc[i] < 1000:
            stage1_circle_1_1['stage1_circle1_distance_speed_collect_result_dam'].iloc[i] = 0
        else:
            pass
    df['stage1_circle1_distance_speed_collect_result_dam'] = stage1_circle_1_1['stage1_circle1_distance_speed_collect_result_dam'
                                                                                                ] + df['thickness_1_collect_result_dam']


    for i in range(len(df)):
        if workmode_collect_1.loc[i, 'workmode_collect_result_dam'] not in [0.000, 7.000]:
            workmode_collect_1.loc[i, 'workmode_collect_result_dam'] = 0

    workmode_collect_1['workmode_collect_result_dam'] = workmode_collect_1['workmode_collect_result_dam'].astype('int')

    for i in range(len(df)):
        if stage1_circle_1_2.loc[i, 'stage1_circle1_distance_speed_collect_result_dam'] in [9000, 5000, 4000]:
            stage1_circle_1_2.loc[i, 'stage1_circle1_distance_speed_collect_result_dam'] = 0

    df['workmode_collect_result_dam'] = workmode_collect_1['workmode_collect_result_dam'
                                                                            ] + stage1_circle_1_2['stage1_circle1_distance_speed_collect_result_dam']


    for i in range(len(df)):
        if workmode_collect_2['workmode_collect_result_dam'].iloc[i] == 7.000:
            workmode_collect_2['workmode_collect_result_dam'].iloc[i] = 0
        else:
            pass

    df['thickness_1_collect_result_dam'] = workmode_collect_2['workmode_collect_result_dam']
###########################################################################################################################



# receip, qty 일치성 (3가지에서 모두 일치하지 않을 때, AbNormal로 나타남.)
train_data['receip_sameness'] = train_data[['receip_no_collect_result_dam',	'receip_no_collect_result_fill1',	'receip_no_collect_result_fill2']].eq(
    train_data[['receip_no_collect_result_dam',	'receip_no_collect_result_fill1',	'receip_no_collect_result_fill2']].iloc[:, 0], axis=0).all(axis=1)
train_data['receip_sameness'] = train_data['receip_sameness'].map({False: 'not_same', True: 'same'})

test_data['receip_sameness'] = test_data[['receip_no_collect_result_dam',	'receip_no_collect_result_fill1',	'receip_no_collect_result_fill2']].eq(
    test_data[['receip_no_collect_result_dam',	'receip_no_collect_result_fill1',	'receip_no_collect_result_fill2']].iloc[:, 0], axis=0).all(axis=1)
test_data['receip_sameness'] = test_data['receip_sameness'].map({False: 'not_same', True: 'same'})

train_data['qty_sameness'] = train_data[['production_qty_collect_result_dam',	'production_qty_collect_result_fill1',	'production_qty_collect_result_fill2']].eq(
    train_data[['production_qty_collect_result_dam',	'production_qty_collect_result_fill1',	'production_qty_collect_result_fill2']].iloc[:, 0], axis=0).all(axis=1)
train_data['qty_sameness'] = train_data['qty_sameness'].map({False: 'not_same', True: 'same'})

test_data['qty_sameness'] = test_data[['production_qty_collect_result_dam',	'production_qty_collect_result_fill1',	'production_qty_collect_result_fill2']].eq(
    test_data[['production_qty_collect_result_dam',	'production_qty_collect_result_fill1',	'production_qty_collect_result_fill2']].iloc[:, 0], axis=0).all(axis=1)
test_data['qty_sameness'] = test_data['qty_sameness'].map({False: 'not_same', True: 'same'})

train_data['equipment_sameness'] = train_data[['equipment_dam', 'equipment_fill1', 'equipment_fill2']].applymap(lambda x: x[-2:]).eq(
    train_data[['equipment_dam', 'equipment_fill1', 'equipment_fill2']].applymap(lambda x: x[-2:]).iloc[:, 0], axis=0).all(axis=1)
train_data['equipment_sameness'] = train_data['equipment_sameness'].map({False: 'not_same', True: 'same'})

test_data['equipment_sameness'] = test_data[['equipment_dam', 'equipment_fill1', 'equipment_fill2']].applymap(lambda x: x[-2:]).eq(
    test_data[['equipment_dam', 'equipment_fill1', 'equipment_fill2']].applymap(lambda x: x[-2:]).iloc[:, 0], axis=0).all(axis=1)
test_data['equipment_sameness'] = test_data['equipment_sameness'].map({False: 'not_same', True: 'same'})

# for df in (train_data, test_data):
#     discharge_time_from_dam = df['discharged_time_of_resin_stage1__collect_result_dam'
#                                         ] + df['discharged_time_of_resin_stage2__collect_result_dam'
#                                                         ] + df['discharged_time_of_resin_stage3__collect_result_dam']

#     df['tact_time_without_discharge_time_dam'] = df['machine_tact_time_collect_result_dam'] - discharge_time_from_dam

In [3]:
train_resin = train_data[[col for col in train_data.columns if not any(keyword in col for keyword in ['autoclave'])]]

test_resin = test_data[[col for col in test_data.columns if not any(keyword in col for keyword in ['autoclave'])]]


# 중복 처리
train_resin.rename(columns={'workorder_dam': 'workorder'}, inplace=True)
train_resin.rename(columns={'model.suffix_dam': 'model.suffix'}, inplace=True)
train_resin.drop(['workorder_fill1', 'workorder_fill2',
                 'model.suffix_fill1', 'model.suffix_fill2'], axis=1, inplace=True)
cols = list(train_resin.columns)
cols.insert(cols.index('workorder') + 1, cols.pop(cols.index('model.suffix')))

test_resin.rename(columns={'workorder_dam': 'workorder'}, inplace=True)
test_resin.rename(columns={'model.suffix_dam': 'model.suffix'}, inplace=True)
test_resin.drop(['workorder_fill1', 'workorder_fill2',
                 'model.suffix_fill1', 'model.suffix_fill2'], axis=1, inplace=True)
cols = list(test_resin.columns)
cols.insert(cols.index('workorder') + 1, cols.pop(cols.index('model.suffix')))

train_resin.rename(columns={'cure_end_position_θ_collect_result_dam': 'cure_start_end_position_θ_collect_result_dam'}, inplace=True)
train_resin.drop(['cure_start_position_θ_collect_result_dam'], axis=1, inplace=True)
test_resin.rename(columns={'cure_end_position_θ_collect_result_dam': 'cure_start_end_position_θ_collect_result_dam'}, inplace=True)
test_resin.drop(['cure_start_position_θ_collect_result_dam'], axis=1, inplace=True)


train_autoclave = train_data[[col for col in train_data.columns if any(keyword in col for keyword in ['autoclave'])]]
train_autoclave = pd.concat([train_autoclave, train_data[['receip_sameness', 'qty_sameness']]], axis=1)
train_autoclave = pd.concat([train_autoclave, train_data['target']], axis=1)
test_autoclave= test_data[[col for col in train_data.columns if any(keyword in col for keyword in ['autoclave'])]]
test_autoclave = pd.concat([test_autoclave, test_data[['receip_sameness', 'qty_sameness']]], axis=1)
test_autoclave = pd.concat([test_autoclave, test_data['target']], axis=1)

In [4]:
train_resin.shape, test_resin.shape, train_autoclave.shape, test_autoclave.shape

((40506, 130), (17361, 130), (40506, 14), (17361, 14))

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef

def get_clf_eval(y_test, y_pred=None):
    confusion = confusion_matrix(y_test, y_pred, labels=[True, False])
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=0)
    recall = recall_score(y_test, y_pred, pos_label=0)
    F1 = f1_score(y_test, y_pred, pos_label=0)

    # Create a DataFrame for better visualization
    confusion_df = pd.DataFrame(confusion, index=['Actual Positive', 'Actual Negative'], columns=['Predicted Positive', 'Predicted Negative'])

    print("오차행렬:\n", confusion_df)
    print("\n정확도: {:.4f}".format(accuracy))
    print("정밀도: {:.4f}".format(precision))
    print("재현율: {:.4f}".format(recall))
    print("F1: {:.4f}".format(F1))

In [6]:
# 쌍 리스트 생성
train_sets = [train_resin, train_autoclave]
test_sets = [test_resin, test_autoclave]

# LightGBM

In [7]:
lgb_pred_list = []
lgb_f1 = []

for train, test in zip(train_sets, test_sets):
    
    # object columns 목록 미리 저장
    obj_cols = train.select_dtypes(include='object').columns.tolist()

    # 'target' 열 제거
    obj_cols = [col for col in obj_cols if col not in ['target']]

    # Label Encoder 생성
    label_encoder = LabelEncoder()

    # target만 Label Encoding 수행
    train['target'] = label_encoder.fit_transform(train['target'])

    for i in obj_cols:
        le = LabelEncoder()
        le=le.fit(train[i])
        train[i]=le.transform(train[i])
        
        for label in np.unique(test[i]):
            if label not in le.classes_: 
                le.classes_ = np.append(le.classes_, label)
        test[i]=le.transform(test[i])
#########################################################################################
    # # object 변환
    # for df in (data, test_data):

    #     for col in obj_cols:
    #         df[col] = df[col].astype('category')


    X_t = train.drop(['target'], axis=1)
    y_t = train[['target']]
    X_test = test.drop(['target'], axis=1)

    tomek = TomekLinks(sampling_strategy='auto')
    X_tomek, y_tomek = tomek.fit_resample(X_t, y_t)
    
    
    skf = StratifiedKFold(n_splits=kfold_num, shuffle=True, random_state=seed_num)
    for train_idx, val_idx in skf.split(X_tomek, y_tomek):
        X_train, X_val = X_tomek.iloc[train_idx], X_tomek.iloc[val_idx]
        y_train, y_val = y_tomek.iloc[train_idx], y_tomek.iloc[val_idx]
###########################################################################################
        counts = list(y_train.value_counts())
        class_weight = {0: counts[0]/sum(counts), 1: counts[1]/sum(counts)}

        lgb_model = LGBMClassifier(objective='binary',
                                n_estimators=5000,
                                class_weight=class_weight,
                                random_state=seed_num) 

        lgb_model.fit(X_train, y_train, 
                    eval_set=[(X_val, y_val)],
                    callbacks=[early_stopping(early_stop)],
                    categorical_feature=obj_cols)

        lgb_pred = lgb_model.predict_proba(X_test, num_iteration=lgb_model.best_iteration_)
        lgb_pred_list.append(lgb_pred)
        
        # Validation set 예측
        lgb_val_pred = lgb_model.predict(X_val)
        lgb_f1.append(f1_score(y_val, lgb_val_pred, pos_label=0))

        # Example usage:
        get_clf_eval(y_val, lgb_val_pred)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 30870, number of negative: 1958
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007471 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4141
[LightGBM] [Info] Number of data points in the train set: 32828, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[810]	valid_0's binary_logloss: 0.289747
오차행렬:
                  Predicted Positive  Predicted Negative
Actual Positiv

Early stopping, best iteration is:
[377]	valid_0's binary_logloss: 0.454073
오차행렬:
                  Predicted Positive  Predicted Negative
Actual Positive                4974                1385
Actual Negative                 232                 160

정확도: 0.7605
정밀도: 0.1036
재현율: 0.4082
F1: 0.1652
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 31793, number of negative: 1959
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001206 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 846
[LightGBM] [Info] Number of data points in the train set: 33752, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=

# Catboost

In [8]:
cat_pred_list = []
cat_f1 = []

for train, test in zip(train_sets, test_sets):
    
    # object columns 목록 미리 저장
    obj_cols = train.select_dtypes(include='object').columns.tolist()

    # 'target' 열 제거
    obj_cols = [col for col in obj_cols if col not in ['target']]

    # Label Encoder 생성
    label_encoder = LabelEncoder()

    # target만 Label Encoding 수행
    train['target'] = label_encoder.fit_transform(train['target'])

    for i in obj_cols:
        le = LabelEncoder()
        le=le.fit(train[i])
        train[i]=le.transform(train[i])
        
        for label in np.unique(test[i]):
            if label not in le.classes_: 
                le.classes_ = np.append(le.classes_, label)
        test[i]=le.transform(test[i])
#########################################################################################
    # object 변환
    for df in (train, test):

        for col in obj_cols:
            df[col] = df[col].astype('category')


    X_t = train.drop(['target'], axis=1)
    y_t = train[['target']]
    X_test = test.drop(['target'], axis=1)

    tomek = TomekLinks(sampling_strategy='auto')
    X_tomek, y_tomek = tomek.fit_resample(X_t, y_t)
    
    
    skf = StratifiedKFold(n_splits=kfold_num, shuffle=True, random_state=seed_num)
    for train_idx, val_idx in skf.split(X_tomek, y_tomek):
        X_train, X_val = X_tomek.iloc[train_idx], X_tomek.iloc[val_idx]
        y_train, y_val = y_tomek.iloc[train_idx], y_tomek.iloc[val_idx]
###########################################################################################
        counts = list(y_train.value_counts())
        class_weight = {0: counts[0]/sum(counts), 1: counts[1]/sum(counts)}

        cat_model = CatBoostClassifier(class_weights=class_weight,
                                    cat_features=obj_cols,
                                    random_state=seed_num)

        cat_model.fit(X_train, y_train, 
                    eval_set = (X_val, y_val), 
                    early_stopping_rounds = early_stop)

        cat_pred = cat_model.predict_proba(X_test)
        cat_pred_list.append(cat_pred)
        
        # Validation set 예측
        cat_val_pred = cat_model.predict(X_val)
        cat_f1.append(f1_score(y_val, cat_val_pred, pos_label=0))

        # Example usage:
        get_clf_eval(y_val, cat_val_pred)


Learning rate set to 0.075062
0:	learn: 0.6847041	test: 0.6876571	best: 0.6876571 (0)	total: 56.4ms	remaining: 56.4s
1:	learn: 0.6780418	test: 0.6832788	best: 0.6832788 (1)	total: 66.5ms	remaining: 33.2s
2:	learn: 0.6727999	test: 0.6800119	best: 0.6800119 (2)	total: 76.1ms	remaining: 25.3s
3:	learn: 0.6688982	test: 0.6774890	best: 0.6774890 (3)	total: 86ms	remaining: 21.4s
4:	learn: 0.6656495	test: 0.6750683	best: 0.6750683 (4)	total: 96.4ms	remaining: 19.2s
5:	learn: 0.6629814	test: 0.6732628	best: 0.6732628 (5)	total: 107ms	remaining: 17.7s
6:	learn: 0.6601182	test: 0.6713642	best: 0.6713642 (6)	total: 116ms	remaining: 16.5s
7:	learn: 0.6579179	test: 0.6701882	best: 0.6701882 (7)	total: 126ms	remaining: 15.6s
8:	learn: 0.6561003	test: 0.6691993	best: 0.6691993 (8)	total: 136ms	remaining: 14.9s
9:	learn: 0.6537133	test: 0.6680781	best: 0.6680781 (9)	total: 145ms	remaining: 14.3s
10:	learn: 0.6518873	test: 0.6671488	best: 0.6671488 (10)	total: 154ms	remaining: 13.9s
11:	learn: 0.650136

44:	learn: 0.6221081	test: 0.6383854	best: 0.6381946 (43)	total: 418ms	remaining: 8.88s
45:	learn: 0.6213503	test: 0.6385914	best: 0.6381946 (43)	total: 429ms	remaining: 8.89s
46:	learn: 0.6207442	test: 0.6385142	best: 0.6381946 (43)	total: 437ms	remaining: 8.86s
47:	learn: 0.6198766	test: 0.6384150	best: 0.6381946 (43)	total: 446ms	remaining: 8.85s
48:	learn: 0.6195187	test: 0.6382216	best: 0.6381946 (43)	total: 456ms	remaining: 8.85s
Stopped by overfitting detector  (5 iterations wait)

bestTest = 0.6381946333
bestIteration = 43

Shrink model to first 44 iterations.
오차행렬:
                  Predicted Positive  Predicted Negative
Actual Positive                4278                1896
Actual Negative                 173                 219

정확도: 0.6849
정밀도: 0.1035
재현율: 0.5587
F1: 0.1747
Learning rate set to 0.075062
0:	learn: 0.6854183	test: 0.6859603	best: 0.6859603 (0)	total: 10.3ms	remaining: 10.3s
1:	learn: 0.6785937	test: 0.6803059	best: 0.6803059 (1)	total: 20ms	remaining: 9.99s


21:	learn: 0.6401984	test: 0.6535908	best: 0.6535908 (21)	total: 220ms	remaining: 9.78s
22:	learn: 0.6391359	test: 0.6530473	best: 0.6530473 (22)	total: 229ms	remaining: 9.74s
23:	learn: 0.6383046	test: 0.6525175	best: 0.6525175 (23)	total: 239ms	remaining: 9.73s
24:	learn: 0.6374475	test: 0.6523874	best: 0.6523874 (24)	total: 249ms	remaining: 9.7s
25:	learn: 0.6367974	test: 0.6519887	best: 0.6519887 (25)	total: 258ms	remaining: 9.68s
26:	learn: 0.6356170	test: 0.6506150	best: 0.6506150 (26)	total: 268ms	remaining: 9.66s
27:	learn: 0.6340595	test: 0.6504253	best: 0.6504253 (27)	total: 277ms	remaining: 9.63s
28:	learn: 0.6332590	test: 0.6499418	best: 0.6499418 (28)	total: 287ms	remaining: 9.62s
29:	learn: 0.6326109	test: 0.6496730	best: 0.6496730 (29)	total: 297ms	remaining: 9.61s
30:	learn: 0.6318602	test: 0.6493858	best: 0.6493858 (30)	total: 306ms	remaining: 9.56s
31:	learn: 0.6308435	test: 0.6497153	best: 0.6493858 (30)	total: 315ms	remaining: 9.52s
32:	learn: 0.6305052	test: 0.6493

22:	learn: 0.6393275	test: 0.6506988	best: 0.6506988 (22)	total: 226ms	remaining: 9.59s
23:	learn: 0.6384071	test: 0.6505377	best: 0.6505377 (23)	total: 236ms	remaining: 9.6s
24:	learn: 0.6377715	test: 0.6502308	best: 0.6502308 (24)	total: 245ms	remaining: 9.57s
25:	learn: 0.6365540	test: 0.6498289	best: 0.6498289 (25)	total: 254ms	remaining: 9.51s
26:	learn: 0.6356775	test: 0.6491404	best: 0.6491404 (26)	total: 264ms	remaining: 9.53s
27:	learn: 0.6349952	test: 0.6486693	best: 0.6486693 (27)	total: 275ms	remaining: 9.53s
28:	learn: 0.6343635	test: 0.6485200	best: 0.6485200 (28)	total: 284ms	remaining: 9.5s
29:	learn: 0.6337524	test: 0.6485333	best: 0.6485200 (28)	total: 293ms	remaining: 9.48s
30:	learn: 0.6327660	test: 0.6482620	best: 0.6482620 (30)	total: 303ms	remaining: 9.46s
31:	learn: 0.6320285	test: 0.6479223	best: 0.6479223 (31)	total: 312ms	remaining: 9.45s
32:	learn: 0.6314933	test: 0.6481783	best: 0.6479223 (31)	total: 322ms	remaining: 9.43s
33:	learn: 0.6310236	test: 0.64803

76:	learn: 0.6156124	test: 0.6493884	best: 0.6481299 (71)	total: 400ms	remaining: 4.79s
Stopped by overfitting detector  (5 iterations wait)

bestTest = 0.6481298899
bestIteration = 71

Shrink model to first 72 iterations.
오차행렬:
                  Predicted Positive  Predicted Negative
Actual Positive                4498                1861
Actual Negative                 191                 201

정확도: 0.6960
정밀도: 0.0975
재현율: 0.5128
F1: 0.1638
Learning rate set to 0.075578
0:	learn: 0.6855153	test: 0.6852877	best: 0.6852877 (0)	total: 5.25ms	remaining: 5.24s
1:	learn: 0.6793326	test: 0.6786370	best: 0.6786370 (1)	total: 10.5ms	remaining: 5.23s
2:	learn: 0.6744743	test: 0.6740992	best: 0.6740992 (2)	total: 15.6ms	remaining: 5.19s
3:	learn: 0.6707330	test: 0.6708983	best: 0.6708983 (3)	total: 20.7ms	remaining: 5.15s
4:	learn: 0.6673300	test: 0.6679220	best: 0.6679220 (4)	total: 25.9ms	remaining: 5.16s
5:	learn: 0.6651711	test: 0.6663238	best: 0.6663238 (5)	total: 31.2ms	remaining: 5.17s
6:

35:	learn: 0.6363444	test: 0.6546385	best: 0.6540056 (33)	total: 179ms	remaining: 4.8s
36:	learn: 0.6358010	test: 0.6542599	best: 0.6540056 (33)	total: 184ms	remaining: 4.78s
37:	learn: 0.6353236	test: 0.6541231	best: 0.6540056 (33)	total: 188ms	remaining: 4.75s
38:	learn: 0.6348119	test: 0.6538936	best: 0.6538936 (38)	total: 192ms	remaining: 4.74s
39:	learn: 0.6341931	test: 0.6535779	best: 0.6535779 (39)	total: 197ms	remaining: 4.72s
40:	learn: 0.6335761	test: 0.6534763	best: 0.6534763 (40)	total: 201ms	remaining: 4.7s
41:	learn: 0.6331146	test: 0.6532805	best: 0.6532805 (41)	total: 205ms	remaining: 4.68s
42:	learn: 0.6323121	test: 0.6531415	best: 0.6531415 (42)	total: 210ms	remaining: 4.66s
43:	learn: 0.6312164	test: 0.6536608	best: 0.6531415 (42)	total: 214ms	remaining: 4.65s
44:	learn: 0.6306938	test: 0.6533886	best: 0.6531415 (42)	total: 218ms	remaining: 4.63s
45:	learn: 0.6301273	test: 0.6530487	best: 0.6530487 (45)	total: 223ms	remaining: 4.62s
46:	learn: 0.6296664	test: 0.65298

36:	learn: 0.6354710	test: 0.6535059	best: 0.6534210 (33)	total: 179ms	remaining: 4.65s
37:	learn: 0.6347361	test: 0.6534516	best: 0.6534210 (33)	total: 183ms	remaining: 4.64s
38:	learn: 0.6341329	test: 0.6531154	best: 0.6531154 (38)	total: 188ms	remaining: 4.63s
39:	learn: 0.6336768	test: 0.6526947	best: 0.6526947 (39)	total: 192ms	remaining: 4.61s
40:	learn: 0.6327674	test: 0.6522529	best: 0.6522529 (40)	total: 197ms	remaining: 4.6s
41:	learn: 0.6321291	test: 0.6521915	best: 0.6521915 (41)	total: 201ms	remaining: 4.58s
42:	learn: 0.6317123	test: 0.6522516	best: 0.6521915 (41)	total: 205ms	remaining: 4.57s
43:	learn: 0.6308590	test: 0.6520924	best: 0.6520924 (43)	total: 210ms	remaining: 4.56s
44:	learn: 0.6301756	test: 0.6514745	best: 0.6514745 (44)	total: 214ms	remaining: 4.54s
45:	learn: 0.6298247	test: 0.6515757	best: 0.6514745 (44)	total: 218ms	remaining: 4.53s
46:	learn: 0.6293571	test: 0.6512087	best: 0.6512087 (46)	total: 222ms	remaining: 4.51s
47:	learn: 0.6288702	test: 0.6514

오차행렬:
                  Predicted Positive  Predicted Negative
Actual Positive                4420                1938
Actual Negative                 196                 196

정확도: 0.6839
정밀도: 0.0918
재현율: 0.5000
F1: 0.1552


# Xgboost

In [9]:
xgb_pred_list = []
xgb_f1 = []

for train, test in zip(train_sets, test_sets):
    
    # object columns 목록 미리 저장
    obj_cols = train.select_dtypes(include='object').columns.tolist()

    # 'target' 열 제거
    obj_cols = [col for col in obj_cols if col not in ['target']]

    # Label Encoder 생성
    label_encoder = LabelEncoder()

    # target만 Label Encoding 수행
    train['target'] = label_encoder.fit_transform(train['target'])

    for i in obj_cols:
        le = LabelEncoder()
        le=le.fit(train[i])
        train[i]=le.transform(train[i])
        
        for label in np.unique(test[i]):
            if label not in le.classes_: 
                le.classes_ = np.append(le.classes_, label)
        test[i]=le.transform(test[i])
#########################################################################################
    # object 변환
    for df in (train, test):

        for col in obj_cols:
            df[col] = df[col].astype('category')


    X_t = train.drop(['target'], axis=1)
    y_t = train[['target']]
    X_test = test.drop(['target'], axis=1)

    tomek = TomekLinks(sampling_strategy='auto')
    X_tomek, y_tomek = tomek.fit_resample(X_t, y_t)
    
    
    skf = StratifiedKFold(n_splits=kfold_num, shuffle=True, random_state=seed_num)
    for train_idx, val_idx in skf.split(X_tomek, y_tomek):
        X_train, X_val = X_tomek.iloc[train_idx], X_tomek.iloc[val_idx]
        y_train, y_val = y_tomek.iloc[train_idx], y_tomek.iloc[val_idx]
###########################################################################################
        xgb_model = XGBClassifier(enable_categorical=True, 
                                n_estimators=5000,
                                scale_pos_weight=len(y_train[y_train['target'] == 0])/len(y_train[y_train['target'] == 1]),
                                early_stopping_rounds=early_stop, 
                                random_state=seed_num,
                                verbose=1)

        xgb_model.fit(X_train, y_train, 
                    eval_set=[(X_val, y_val)])

        xgb_pred = xgb_model.predict_proba(X_test)
        xgb_pred_list.append(xgb_pred)
        
        # Validation set 예측
        xgb_val_pred = xgb_model.predict(X_val)
        xgb_f1.append(f1_score(y_val, xgb_val_pred, pos_label=0))

        # Example usage:
        get_clf_eval(y_val, xgb_val_pred)


[0]	validation_0-logloss:0.66716
[1]	validation_0-logloss:0.65171
[2]	validation_0-logloss:0.64113
[3]	validation_0-logloss:0.63365
[4]	validation_0-logloss:0.62681
[5]	validation_0-logloss:0.62227
[6]	validation_0-logloss:0.61607
[7]	validation_0-logloss:0.61277
[8]	validation_0-logloss:0.61129
[9]	validation_0-logloss:0.60910
[10]	validation_0-logloss:0.60600
[11]	validation_0-logloss:0.60485
[12]	validation_0-logloss:0.60388
[13]	validation_0-logloss:0.60253
[14]	validation_0-logloss:0.60191
[15]	validation_0-logloss:0.60084
[16]	validation_0-logloss:0.59517
[17]	validation_0-logloss:0.59428
[18]	validation_0-logloss:0.59328
[19]	validation_0-logloss:0.59281
[20]	validation_0-logloss:0.58968
[21]	validation_0-logloss:0.58912
[22]	validation_0-logloss:0.58867
[23]	validation_0-logloss:0.58762
[24]	validation_0-logloss:0.58547
[25]	validation_0-logloss:0.58353
[26]	validation_0-logloss:0.58088
[27]	validation_0-logloss:0.58042
[28]	validation_0-logloss:0.57888
[29]	validation_0-loglos

[238]	validation_0-logloss:0.41905
[239]	validation_0-logloss:0.41884
[240]	validation_0-logloss:0.41768
[241]	validation_0-logloss:0.41677
[242]	validation_0-logloss:0.41567
[243]	validation_0-logloss:0.41559
[244]	validation_0-logloss:0.41477
[245]	validation_0-logloss:0.41459
[246]	validation_0-logloss:0.41454
[247]	validation_0-logloss:0.41453
[248]	validation_0-logloss:0.41433
[249]	validation_0-logloss:0.41372
[250]	validation_0-logloss:0.41353
[251]	validation_0-logloss:0.41345
[252]	validation_0-logloss:0.41320
[253]	validation_0-logloss:0.41288
[254]	validation_0-logloss:0.41260
[255]	validation_0-logloss:0.41187
[256]	validation_0-logloss:0.41167
[257]	validation_0-logloss:0.41139
[258]	validation_0-logloss:0.41062
[259]	validation_0-logloss:0.41044
[260]	validation_0-logloss:0.41008
[261]	validation_0-logloss:0.41000
[262]	validation_0-logloss:0.40947
[263]	validation_0-logloss:0.40902
[264]	validation_0-logloss:0.40862
[265]	validation_0-logloss:0.40837
[266]	validation_0-l

[473]	validation_0-logloss:0.35687
[474]	validation_0-logloss:0.35635
[475]	validation_0-logloss:0.35621
[476]	validation_0-logloss:0.35588
[477]	validation_0-logloss:0.35569
[478]	validation_0-logloss:0.35536
[479]	validation_0-logloss:0.35535
[480]	validation_0-logloss:0.35525
[481]	validation_0-logloss:0.35524
[482]	validation_0-logloss:0.35497
[483]	validation_0-logloss:0.35506
[484]	validation_0-logloss:0.35448
[485]	validation_0-logloss:0.35447
[486]	validation_0-logloss:0.35446
[487]	validation_0-logloss:0.35452
[488]	validation_0-logloss:0.35444
[489]	validation_0-logloss:0.35441
[490]	validation_0-logloss:0.35392
[491]	validation_0-logloss:0.35378
[492]	validation_0-logloss:0.35367
[493]	validation_0-logloss:0.35355
[494]	validation_0-logloss:0.35348
[495]	validation_0-logloss:0.35353
[496]	validation_0-logloss:0.35308
[497]	validation_0-logloss:0.35304
[498]	validation_0-logloss:0.35287
[499]	validation_0-logloss:0.35259
[500]	validation_0-logloss:0.35226
[501]	validation_0-l

[42]	validation_0-logloss:0.55886
[43]	validation_0-logloss:0.55567
[44]	validation_0-logloss:0.55465
[45]	validation_0-logloss:0.55395
[46]	validation_0-logloss:0.55209
[47]	validation_0-logloss:0.55138
[48]	validation_0-logloss:0.55064
[49]	validation_0-logloss:0.54934
[50]	validation_0-logloss:0.54901
[51]	validation_0-logloss:0.54846
[52]	validation_0-logloss:0.54817
[53]	validation_0-logloss:0.54699
[54]	validation_0-logloss:0.54394
[55]	validation_0-logloss:0.54118
[56]	validation_0-logloss:0.53950
[57]	validation_0-logloss:0.53856
[58]	validation_0-logloss:0.53656
[59]	validation_0-logloss:0.53519
[60]	validation_0-logloss:0.53442
[61]	validation_0-logloss:0.53266
[62]	validation_0-logloss:0.53204
[63]	validation_0-logloss:0.53071
[64]	validation_0-logloss:0.53020
[65]	validation_0-logloss:0.52892
[66]	validation_0-logloss:0.52846
[67]	validation_0-logloss:0.52779
[68]	validation_0-logloss:0.52782
[69]	validation_0-logloss:0.52721
[70]	validation_0-logloss:0.52631
[71]	validatio

[278]	validation_0-logloss:0.40612
[279]	validation_0-logloss:0.40589
[280]	validation_0-logloss:0.40508
[281]	validation_0-logloss:0.40481
[282]	validation_0-logloss:0.40494
[283]	validation_0-logloss:0.40388
[284]	validation_0-logloss:0.40336
[285]	validation_0-logloss:0.40339
[286]	validation_0-logloss:0.40331
[287]	validation_0-logloss:0.40289
[288]	validation_0-logloss:0.40292
[289]	validation_0-logloss:0.40274
[290]	validation_0-logloss:0.40239
[291]	validation_0-logloss:0.40238
[292]	validation_0-logloss:0.40193
[293]	validation_0-logloss:0.40188
[294]	validation_0-logloss:0.40191
[295]	validation_0-logloss:0.40163
[296]	validation_0-logloss:0.40131
[297]	validation_0-logloss:0.40108
[298]	validation_0-logloss:0.40077
[299]	validation_0-logloss:0.40056
[300]	validation_0-logloss:0.40010
[301]	validation_0-logloss:0.39989
[302]	validation_0-logloss:0.39917
[303]	validation_0-logloss:0.39845
[304]	validation_0-logloss:0.39787
[305]	validation_0-logloss:0.39732
[306]	validation_0-l

...

[127]	validation_0-logloss:0.53259
[128]	validation_0-logloss:0.53244
[129]	validation_0-logloss:0.53232
[130]	validation_0-logloss:0.53191
[131]	validation_0-logloss:0.53119
[132]	validation_0-logloss:0.53053
[133]	validation_0-logloss:0.53015
[134]	validation_0-logloss:0.53004
[135]	validation_0-logloss:0.52981
[136]	validation_0-logloss:0.52947
[137]	validation_0-logloss:0.52892
[138]	validation_0-logloss:0.52829
[139]	validation_0-logloss:0.52797
[140]	validation_0-logloss:0.52763
[141]	validation_0-logloss:0.52742
[142]	validation_0-logloss:0.52690
[143]	validation_0-logloss:0.52673
[144]	validation_0-logloss:0.52660
[145]	validation_0-logloss:0.52620
[146]	validation_0-logloss:0.52548
[147]	validation_0-logloss:0.52389
[148]	validation_0-logloss:0.52334
[149]	validation_0-logloss:0.52291
[150]	validation_0-logloss:0.52255
[151]	validation_0-logloss:0.52214
[152]	validation_0-logloss:0.52130
[153]	validation_0-logloss:0.52071
[154]	validation_0-logloss:0.52049
[155]	validation_0-l

[11]	validation_0-logloss:0.62779
[12]	validation_0-logloss:0.62691
[13]	validation_0-logloss:0.62656
[14]	validation_0-logloss:0.62368
[15]	validation_0-logloss:0.62310
[16]	validation_0-logloss:0.62052
[17]	validation_0-logloss:0.62007
[18]	validation_0-logloss:0.61903
[19]	validation_0-logloss:0.61762
[20]	validation_0-logloss:0.61722
[21]	validation_0-logloss:0.61588
[22]	validation_0-logloss:0.61472
[23]	validation_0-logloss:0.61351
[24]	validation_0-logloss:0.61202
[25]	validation_0-logloss:0.61144
[26]	validation_0-logloss:0.61066
[27]	validation_0-logloss:0.60883
[28]	validation_0-logloss:0.60724
[29]	validation_0-logloss:0.60620
[30]	validation_0-logloss:0.60517
[31]	validation_0-logloss:0.60334
[32]	validation_0-logloss:0.60153
[33]	validation_0-logloss:0.59981
[34]	validation_0-logloss:0.59775
[35]	validation_0-logloss:0.59530
[36]	validation_0-logloss:0.59459
[37]	validation_0-logloss:0.59389
[38]	validation_0-logloss:0.59367
[39]	validation_0-logloss:0.59244
[40]	validatio

[248]	validation_0-logloss:0.50176
[249]	validation_0-logloss:0.50152
[250]	validation_0-logloss:0.50142
[251]	validation_0-logloss:0.50132
[252]	validation_0-logloss:0.50130
[253]	validation_0-logloss:0.50109
[254]	validation_0-logloss:0.50099
[255]	validation_0-logloss:0.50093
[256]	validation_0-logloss:0.50073
[257]	validation_0-logloss:0.50031
[258]	validation_0-logloss:0.50032
[259]	validation_0-logloss:0.50022
[260]	validation_0-logloss:0.50001
[261]	validation_0-logloss:0.49996
[262]	validation_0-logloss:0.49986
[263]	validation_0-logloss:0.49956
[264]	validation_0-logloss:0.49916
[265]	validation_0-logloss:0.49922
[266]	validation_0-logloss:0.49921
[267]	validation_0-logloss:0.49908
[268]	validation_0-logloss:0.49904
[269]	validation_0-logloss:0.49899
[270]	validation_0-logloss:0.49902
[271]	validation_0-logloss:0.49861
[272]	validation_0-logloss:0.49867
[273]	validation_0-logloss:0.49820
[274]	validation_0-logloss:0.49800
[275]	validation_0-logloss:0.49771
[276]	validation_0-l

[120]	validation_0-logloss:0.53289
[121]	validation_0-logloss:0.53246
[122]	validation_0-logloss:0.53225
[123]	validation_0-logloss:0.53198
[124]	validation_0-logloss:0.53180
[125]	validation_0-logloss:0.53108
[126]	validation_0-logloss:0.53067
[127]	validation_0-logloss:0.53047
[128]	validation_0-logloss:0.53017
[129]	validation_0-logloss:0.53001
[130]	validation_0-logloss:0.52966
[131]	validation_0-logloss:0.52945
[132]	validation_0-logloss:0.52939
[133]	validation_0-logloss:0.52935
[134]	validation_0-logloss:0.52914
[135]	validation_0-logloss:0.52916
[136]	validation_0-logloss:0.52815
[137]	validation_0-logloss:0.52708
[138]	validation_0-logloss:0.52706
[139]	validation_0-logloss:0.52702
[140]	validation_0-logloss:0.52568
[141]	validation_0-logloss:0.52544
[142]	validation_0-logloss:0.52516
[143]	validation_0-logloss:0.52469
[144]	validation_0-logloss:0.52452
[145]	validation_0-logloss:0.52442
[146]	validation_0-logloss:0.52424
[147]	validation_0-logloss:0.52316
[148]	validation_0-l

[355]	validation_0-logloss:0.47655
[356]	validation_0-logloss:0.47653
[357]	validation_0-logloss:0.47650
[358]	validation_0-logloss:0.47632
[359]	validation_0-logloss:0.47632
[360]	validation_0-logloss:0.47599
[361]	validation_0-logloss:0.47575
[362]	validation_0-logloss:0.47572
[363]	validation_0-logloss:0.47534
[364]	validation_0-logloss:0.47529
[365]	validation_0-logloss:0.47519
[366]	validation_0-logloss:0.47528
[367]	validation_0-logloss:0.47500
[368]	validation_0-logloss:0.47489
[369]	validation_0-logloss:0.47478
[370]	validation_0-logloss:0.47469
[371]	validation_0-logloss:0.47454
[372]	validation_0-logloss:0.47441
[373]	validation_0-logloss:0.47440
[374]	validation_0-logloss:0.47394
[375]	validation_0-logloss:0.47393
[376]	validation_0-logloss:0.47387
[377]	validation_0-logloss:0.47366
[378]	validation_0-logloss:0.47350
[379]	validation_0-logloss:0.47331
[380]	validation_0-logloss:0.47335
[381]	validation_0-logloss:0.47305
[382]	validation_0-logloss:0.47289
[383]	validation_0-l

# Score

In [10]:
average = 0
for t in lgb_f1:
    average += t
average = (average)/(kfold_num*2)
round(average, 4)

0.198

In [11]:
average = 0
for t in cat_f1:
    average += t
average = (average)/(kfold_num*2)
round(average, 4)

0.1662

In [12]:
average = 0
for t in xgb_f1:
    average += t
average = (average)/(kfold_num*2)
round(average, 4)

0.185

# Ensemble

In [13]:
# 확률값 np.array로 변환
cat_pred_list = np.array(cat_pred_list)
lgb_pred_list = np.array(lgb_pred_list)
xgb_pred_list = np.array(xgb_pred_list)

In [14]:
# 최종 예측 정리
pred_proba = np.zeros((17361, 2))

for elem in cat_pred_list + lgb_pred_list + xgb_pred_list:
    pred_proba += elem

sum_proba = pd.DataFrame((pred_proba)/((kfold_num*2)*3))

sum_proba['predicted'] = sum_proba.apply(lambda row: 0 if row[0] > row[1] else 1, axis=1)

submission['target'] = sum_proba['predicted']
submission['target'] = submission['target'].map({0: 'AbNormal', 1: 'Normal'})

submission['target'].value_counts()

target
Normal      15797
AbNormal     1564
Name: count, dtype: int64

In [ ]:
# 제출 파일 저장
submission.to_csv("./submission.csv", index=False)